In [1]:
import pickle
import time
import numpy as np
import pandas as pd
import h5py
import os

In [4]:
filename = "./xian01_part113.h5"
num = 0

In [6]:
# 打开HDF5文件
with h5py.File(filename, 'r') as file:
    # 列出所有顶层keys
    keys = file.keys()
    print(keys)

<KeysViewHDF5 ['road_info', 'trip_info', 'trips']>


In [5]:
# 读取 HDF5 文件
df_trips = pd.read_hdf(filename, key='trips')

# 打印数据框
df_trips.shape

(1672279, 10)

In [10]:
df_trips

,seq_i,road,obs,obs_ne,timestamp,longitude,latitude,length,road_prop,trip
0,0,5400,0,0,2018-10-13 09:34:59,109.00838,34.25118,474.701,0.307591,1130000
1,1,5400,1,0,2018-10-13 09:35:05,109.00803,34.25114,474.701,0.238429,1130000
2,2,5400,2,0,2018-10-13 09:35:11,109.00764,34.25117,474.701,0.162274,1130000
3,3,5400,3,0,2018-10-13 09:35:17,109.00762,34.25118,474.701,0.158472,1130000
4,4,5397,4,0,2018-10-13 09:35:23,109.00763,34.25118,6.674,1.000000,1130000
...,...,...,...,...,...,...,...,...,...,...
100,100,5217,100,0,2018-10-14 13:50:44,108.98879,34.21851,51.352,1.000000,1139999
101,101,5217,101,0,2018-10-14 13:50:50,108.98851,34.21828,51.352,1.000000,1139999
102,102,2944,102,0,2018-10-14 13:50:55,108.98842,34.21822,310.467,1.000000,1139999
103,103,2944,103,0,2018-10-14 13:50:59,108.98842,34.21822,310.467,1.000000,1139999


In [ ]:
num = 0
def generate_tr_data(filename,num):
    start_time = time.time()
    print(f"开始运行：{start_time}秒",filename)
    with h5py.File(filename,'r') as file:
        keys = list(file.keys())    
    df_road = pd.read_hdf(filename,key=keys[0]) ##路网信息
    df_trip_info = pd.read_hdf(filename,key=keys[1]) ##轨迹信息
    df_trips = pd.read_hdf(filename,key=keys[2]) ##轨迹点信息
    tr_lists = []
    tr_list = []
    i = df_trips.iloc[0][1]
    v = df_trip_info.iloc[0][0]
    k=0
    j =0
    csv_list = [] 
    link_distance=[]
    link_id = []
    link_time= []
    filename = f"list_{num}.npy"
    for point in df_trips.iterrows():  ## 路段list
        if i == point[1][1]: ## 路段号一样的
            x = [point[1][1],point[1][4],point[1][7],point[1][-1]] # 路段， 时间戳， 路段长度， 轨迹ID
            tr_list.append(x)
        elif v == tr_list[0][-1]: ##同一路径的
            link_distance.append(round(tr_list[0][2], 3)) #保留3位小数的路段长度
            link_id.append(tr_list[0][0])
            link_time.append((tr_list[-1][1] - tr_list[0][1]).total_seconds()) #路段通行时间最后一个减第一个
            tr_list = []
            x = [point[1][1],point[1][4],point[1][7],point[1][-1]]
            tr_list.append(x)
            i = point[1][1]
        elif v != tr_list[0][-1] :##不同路径的
            start = df_trip_info.iloc[j][1]
            driver = int(df_trip_info.iloc[j][-1], 16)
            link_num = len(link_distance)
            cross_num = link_num-1
            label = (df_trip_info.iloc[j][2]-df_trip_info.iloc[j][1]).total_seconds() 
            distance = df_trip_info.iloc[j][3]*1000  ##km转m
            csv_list.append([start,driver,link_distance,distance,link_id,link_num,cross_num,link_time,label])
            tr_list = []
            link_distance=[]
            link_id = []
            link_time= []
            x = [point[1][1],point[1][4],point[1][7],point[1][-1]]
            tr_list.append(x)
            i = point[1][1]
            j = j + 1
            v = v + 1
        k=k+1
        
     
    name = ["start", "driver", "link_distance", "distance", "link_id", "link_num","cross_num","link_time","label"] #处理后的特征
    test = pd.DataFrame(columns=name,data=csv_list)
    array = test.to_numpy()
    subdirectory = "xian"
    filepath = os.path.join(subdirectory, filename)
    print("文件：",filepath)
    np.save(filepath, array)
    print("轨迹文件已生成:",num)
    end_time = time.time()
    execution_time = end_time - start_time
    print(f"执行时间：{execution_time}秒")
    return


In [ ]:
def get_all_filenames(directory):
    filenames = []
    for root, dirs, files in os.walk(directory):
        for file in files:
            if file.endswith(".h5"):
                filename = os.path.join(root, file)
                filenames.append(filename)
    return filenames

# 指定要搜索的文件夹路径
directory = './'

# 获取文件夹中所有文件的文件名
filenames = get_all_filenames(directory)

for filename in filenames:
    generate_tr_data(filename,num)
    num += 1

# dataset

In [ ]:
seq_data = np.load(filepath + 'train.npy', allow_pickle=True)

In [ ]:
# 构建Dataset
class GiscupDataset(torch.utils.data.Dataset):
    def __init__(self, seq_data, FLAGS, device):
        all_num = []
        all_id = []
        all_time = []
        all_flow = []
        all_label = []
        all_cross = []
        padding_size = FLAGS.segment_num
        drivers_num = FLAGS.drivers_num
        print(seq_data.shape)
        seq_data[:, 9] = seq_data[:, 9]-1
        for i in range(len(seq_data)):
            length = len(seq_data[i][4])
            #             all_num.append(length)  # list 500
            ids = seq_data[i][4] + [-1] * (padding_size - length)
            all_id.append(ids)  # link id

            time = seq_data[i][12] + [-1] * (padding_size - length)
            all_time.append(time)  # list

            flow = seq_data[i][11]+[-1]*(padding_size - length)
            all_flow.append(flow)  # list

            all_num.append(seq_data[i][5])  # link num porto-1记住
            all_cross.append(seq_data[i][6])  # cross num

            all_label.append(seq_data[i][8])  # label
            # all_s = [slices[i]] * padding_size
            # all_slice.append(all_s)
        self.all_num = torch.tensor(all_num, dtype=torch.int)
        self.all_id = torch.tensor(all_id, dtype=torch.long) + 1
        self.all_real = torch.tensor(all_time, dtype=torch.float)
        self.all_flow = torch.tensor(all_flow, dtype=torch.int)
        self.targets = torch.tensor(all_label) + 1.0
        wide_deep_raw = torch.tensor(seq_data[:, [1, 9, 3, 5, 6]].astype(float))  # driver_id slice_window distance link_num cross_num
        # wide_deep_raw = torch.tensor(seq_data[:, [1, 11, 3, 5]].astype(float))  # driver_id distance link_num
        self.deep_category = wide_deep_raw[:, :2].long()  # driver_id slice_window
        self.deep_category = self.deep_category.long()
        #         self.deep_real = wide_deep_raw[:, 2:].float1()
        self.deep_real = wide_deep_raw[:, 2:].float()  # distance link_num cross_num
        self.wide_index = wide_deep_raw.clone()  # [256, 5]
        #         self.wide_index = copy.deepcopy(wide_deep_raw)  # [256, 4]
        self.wide_index[:, 2:] = 0
        #         self.wide_index[:, 2:] = 0
        self.wide_index += torch.tensor([0, drivers_num, drivers_num + 288, drivers_num + 288 + 1, drivers_num + 288 + 1 + 1])  # WDR-LC 5个
        # self.wide_index += torch.tensor([0, drivers_num, drivers_num + 288, drivers_num + 288 + 1])  # WDR 4个
        self.wide_index = self.wide_index.long()
        self.wide_value = wide_deep_raw.float()
        #         self.wide_value = wide_deep_raw.float()
        self.wide_value[:, :2] = 1.0  # 类别特征的wide value 为1，只要其embedding后的值，连续特征的wide_value为连续值

    def __getitem__(self, index):
        return self.wide_index[index], self.wide_value[index], self.deep_category[index], self.deep_real[index], \
               self.all_id[index], self.all_num[index], self.all_real[index], self.all_flow[index], self.targets[index]

    def __len__(self):
        return self.targets.shape[0]